# Lab19 - Evaluación de Algoritmos
**Extracción de conocimiento en Bases de Datos**

**Objetivo:** Evaluar distintos algoritmos de clasificación para determinar cuál ofrece el mejor desempeño antes de integrarlo a un sistema.

**Requisitos:**
1. **Repositorio:** GitHub (cuenta personal del alumno).
2. **Directorio:** El archivo debe estar dentro de una carpeta llamada `Notebooks`.
3. **Nombre del archivo:** `Lab18.ipynb`.
4. **Contenido obligatorio:** El notebook debe finalizar con una celda de Markdown titulada "Conclusiones", donde el estudiante reflexione sobre los hallazgos y las decisiones tomadas durante la limpieza.

In [3]:
# Paso 1: Importación de librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Herramientas de Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Librería para cargar dataset
import kagglehub

## Paso 2: Carga del dataset y Exploración
Para esta evaluación utilizaremos el dataset del Titanic, ya que nos permite aplicar técnicas de limpieza antes de clasificar.

In [4]:
# Descargar y cargar el dataset
path = kagglehub.dataset_download("yasserh/titanic-dataset")
df = pd.read_csv(path + "/Titanic-Dataset.csv")

# Exploración inicial
print(df.info())
print(df.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB
None
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int6

## Paso 3: Limpieza y Tratamiento de Datos
Se toman decisiones clave para manejar los valores nulos y eliminar columnas que no aportan información al modelo predictivo.

In [5]:
# 1. Tratamiento de nulos
# Se rellena la edad con la mediana para no afectar la distribución por valores atípicos
df['Age'] = df['Age'].fillna(df['Age'].median())

# Se rellena el puerto de embarque con la moda (el valor más frecuente)
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

# 2. Eliminación de columnas irrelevantes o con demasiados nulos (Cabin)
df = df.drop(columns=['Cabin', 'PassengerId', 'Name', 'Ticket'])

# 3. Transformación de variables categóricas (One-Hot Encoding)
df = pd.get_dummies(df, columns=['Sex', 'Embarked'], drop_first=True)

df.head()

,Survived,Pclass,Age,SibSp,Parch,Fare,Sex_male,Embarked_Q,Embarked_S
0,0,3,22.0,1,0,7.2500,True,False,True
1,1,1,38.0,1,0,71.2833,False,False,False
2,1,3,26.0,0,0,7.9250,False,False,True
3,1,1,35.0,1,0,53.1000,False,False,True
4,0,3,35.0,0,0,8.0500,True,False,True


## Paso 4: División y Escalado de Datos
Separamos la variable objetivo (`Survived`) de las predictoras y escalamos los datos, un paso fundamental para algoritmos basados en distancias como KNN.

In [6]:
# Definir variables X e y
X = df.drop(columns=['Survived'])
y = df['Survived']

# Train/Test Split (80% entrenamiento, 20% prueba)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

# Escalado de datos
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## Paso 5: Entrenamiento y Evaluación de Algoritmos
Entrenaremos tres algoritmos diferentes (Árbol de Decisión, KNN y Random Forest) para comparar su rendimiento.

In [7]:
# 1. Árbol de Decisión (CART)
modelo_dt = DecisionTreeClassifier(max_depth=4, random_state=42)
modelo_dt.fit(X_train_scaled, y_train)
pred_dt = modelo_dt.predict(X_test_scaled)
acc_dt = accuracy_score(y_test, pred_dt)

# 2. K-Nearest Neighbors (KNN)
modelo_knn = KNeighborsClassifier(n_neighbors=5)
modelo_knn.fit(X_train_scaled, y_train)
pred_knn = modelo_knn.predict(X_test_scaled)
acc_knn = accuracy_score(y_test, pred_knn)

# 3. Random Forest
modelo_rf = RandomForestClassifier(n_estimators=100, random_state=42)
modelo_rf.fit(X_train_scaled, y_train)
pred_rf = modelo_rf.predict(X_test_scaled)
acc_rf = accuracy_score(y_test, pred_rf)

# Mostrar resultados comparativos
resultados = pd.DataFrame({
    'Algoritmo': ['Árbol de Decisión', 'KNN', 'Random Forest'],
    'Accuracy': [acc_dt, acc_knn, acc_rf]
})

print(resultados.sort_values(by='Accuracy', ascending=False))

           Algoritmo  Accuracy
2      Random Forest  0.821229
1                KNN  0.804469
0  Árbol de Decisión  0.798883


# Conclusiones

Al finalizar esta práctica, he podido evaluar y contrastar el desempeño de múltiples algoritmos, llegando a las siguientes reflexiones:

* **Decisiones de Limpieza:** El tratamiento adecuado de los valores nulos fue vital. Reemplazar la edad con la mediana en lugar de la media protegió al modelo de los sesgos introducidos por valores atípicos. Asimismo, la eliminación de la columna `Cabin` fue una decisión necesaria, ya que contenía demasiados valores faltantes y hubiera introducido "ruido" perjudicial, especialmente para algoritmos sensibles a la dimensionalidad como KNN. El escalado posterior garantizó que variables con magnitudes altas (como `Fare`) no dominaran el cálculo de distancias.
* **Hallazgos sobre los Modelos:** Al comparar los algoritmos, **Random Forest** obtuvo generalmente el mejor rendimiento en términos de *Accuracy*. Esto se debe a su naturaleza de "ensamble", que construye múltiples árboles y promedia sus resultados, reduciendo significativamente el sobreajuste (overfitting) que suele presentarse en un único Árbol de Decisión. Por su parte, KNN demostró ser competitivo gracias al escalado correcto de las variables, aunque requirió un mayor esfuerzo computacional al calcular distancias en el conjunto de prueba.
* En un escenario real de integración a un sistema, elegiría Random Forest por su solidez y capacidad de generalización ante datos no vistos.